# EDA 시각화 — Team3 빡빡이

제출 코드(`5th_datathon_code_빡빡이.ipynb`)와 **별개**인 읽기 전용 분석 노트북이다. `./epoch_data/`만 있으면 실행되고, 그림 3개를 PNG로 저장한다.

1. 08-25~27 플랫폼 전체 조회수 급등
2. 상위 20% 문턱의 이동 (분포 이동)
3. 결측은 무작위가 아니다 (결측 여부 예측 AUC)

In [ ]:
# EDA 시각화 (제출 코드와 별개, 읽기 전용 분석)
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
for f in ['Malgun Gothic','AppleGothic','NanumGothic','Noto Sans CJK KR','Noto Sans CJK JP']:
    if any(f == x.name for x in font_manager.fontManager.ttflist): plt.rcParams['font.family']=f; break
plt.rcParams['axes.unicode_minus']=False
GREEN,GOLD,GREY,INK,LINE='#37654A','#C28B2C','#8A8F8B','#262626','#D9E2DC'
DATA='./epoch_data/'
v=pd.read_csv(DATA+'video_5d_views.csv'); v['published_at']=pd.to_datetime(v.published_at,utc=True).dt.tz_localize(None)
L=v[(v.is_shorts==0)&v.view_5d.notna()].copy(); L['lv']=np.log1p(L.view_5d)
def style(ax):
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['left','bottom']: ax.spines[s].set_color(LINE)
    ax.tick_params(colors=GREY, labelsize=11); ax.grid(axis='y',color='#EEF1EE',lw=0.8); ax.set_axisbelow(True)


### ① 08-25~27 급등: 업로드일별 '채널 평소(08-20 이전 중앙값) 대비' 로그 조회수의 중앙값

In [ ]:
# ① 08-25~27 급등: 업로드일별 '채널 평소(08-20 이전 중앙값) 대비' 로그 조회수의 중앙값
pre=L[L.published_at<'2026-08-20'].groupby('channel_id').lv.median()
L['rel']=L.lv-L.channel_id.map(pre)
daily=L.dropna(subset=['rel']).groupby(L.published_at.dt.normalize()).rel.median().loc['2026-07-20':]
fig,ax=plt.subplots(figsize=(10,4.2),dpi=200); style(ax)
ax.plot(daily.index,daily.values,color=GREEN,lw=2.2)
ax.axvspan(pd.Timestamp('2026-08-25'),pd.Timestamp('2026-08-27'),color=GOLD,alpha=0.15,lw=0)
ax.axhline(0,color=GREY,lw=0.8,ls='--')
ax.annotate('08-25~27 급등\n평소 대비 약 2.2배',xy=(pd.Timestamp('2026-08-28'),daily.loc['2026-08-28']),xytext=(pd.Timestamp('2026-08-08'),0.62),
            fontsize=12,color=INK,arrowprops=dict(arrowstyle='->',color=GREY))
ax.set_ylabel('평소 대비 로그 조회수 (중앙값)',color=GREY,fontsize=11)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%m-%d'))
fig.tight_layout(); fig.savefig('eda_surge.png',transparent=False,facecolor='white'); plt.close()


### ② 상위 20% 문턱의 이동: 기준일별 s의 80% 분위수 (미래 창 18일이 모두 관측되는 날짜)

In [ ]:
# ② 상위 20% 문턱의 이동: 기준일별 s의 80% 분위수 (미래 창 18일이 모두 관측되는 날짜)
def thr(D,F=18):
    P=L[L.published_at<D].groupby('channel_id').view_5d.agg(['median','size']); P=P[(P['size']>=3)&(P['median']>=100)]['median']
    Q=L[(L.published_at>=D)&(L.published_at<D+pd.Timedelta(days=F))].groupby('channel_id').view_5d.median().reindex(P.index)
    return (np.log1p(Q.fillna(0))-np.log1p(P)).quantile(.8)
T=pd.Series({D:thr(D) for D in pd.date_range('2026-07-18','2026-08-13')})
fig,ax=plt.subplots(figsize=(10,4.2),dpi=200); style(ax)
ax.plot(T.index,T.values,color=GREEN,lw=2.2,marker='o',ms=4)
ax.annotate(f'{T.iloc[0]:.2f}',(T.index[0],T.iloc[0]),textcoords='offset points',xytext=(0,10),ha='center',fontsize=12,color=INK)
ax.annotate(f'{T.iloc[-1]:.2f}',(T.index[-1],T.iloc[-1]),textcoords='offset points',xytext=(0,10),ha='center',fontsize=12,color=INK,fontweight='bold')
ax.set_ylabel('상위 20% 문턱 (s의 80% 분위수)',color=GREY,fontsize=11)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%m-%d'))
fig.tight_layout(); fig.savefig('eda_threshold.png',facecolor='white'); plt.close()


### ③ 결측은 무작위가 아니다: 결측 여부를 다른 (순위) 피처로 예측한 AUC

In [ ]:
# ③ 결측은 무작위가 아니다: 결측 여부를 다른 (순위) 피처로 예측한 AUC
#    run_final.py 의 feats() 와 같은 정의로 기준일별 피처를 다시 계산 (제출 코드는 수정하지 않음)
def feats(D):
    pl=L[L.published_at<D].copy(); pl['age']=(D-pl.published_at)/pd.Timedelta(days=1); g=pl.groupby('channel_id')
    f=pd.DataFrame({'past_med':g.lv.median(),'past_std':g.lv.std(),'past_n':g.size()})
    for w in (3,7,14):
        r=pl[pl.age<=w].groupby('channel_id').lv; f[f'm{w}']=r.median()-f.past_med; f[f'r{w}_n']=r.size()
    f['r7_n']=f.r7_n.fillna(0); f['r14_n']=f.r14_n.fillna(0)
    f['m14_old']=pl[pl.age<=14].groupby('channel_id').lv.median()-pl[pl.age>14].groupby('channel_id').lv.median()
    f['days_since']=g.age.min(); f['upl_rate']=f.past_n/g.age.max().clip(lower=1)
    f['cv_last10']=g.tail(10).groupby('channel_id').lv.std()
    return f[(f.past_n>=3)].assign(D=D)
X=pd.concat([feats(D) for D in pd.date_range('2026-07-25','2026-08-31',freq='3D')])
base=['past_med','past_std','r7_n','r14_n','days_since','upl_rate','cv_last10']
for c in base: X['rk_'+c]=X.groupby('D')[c].rank(pct=True)
res={}
for c,lab in [('m3','최근 3일 모멘텀 (m3)'),('m7','최근 7일 모멘텀 (m7)'),('m14_old','2주 vs 이전 (m14_old)')]:
    y=X[c].isna().astype(int); Z=X[['rk_'+b for b in base]].fillna(0.5)
    res[lab]=(y.mean(),roc_auc_score(y,LogisticRegression(max_iter=2000).fit(Z,y).predict_proba(Z)[:,1]))
R=pd.DataFrame(res,index=['rate','auc']).T.sort_values('auc')
fig,ax=plt.subplots(figsize=(10,3.6),dpi=200); style(ax); ax.grid(axis='x',color='#EEF1EE'); ax.grid(axis='y',visible=False)
ax.barh(R.index,R.auc,color=GREEN,height=0.5)
ax.axvline(0.5,color=GOLD,lw=1.5,ls='--'); ax.text(0.51,len(R)-0.45,'무작위 결측이면 0.5',color=GOLD,fontsize=11,va='bottom')
for i,(a,r) in enumerate(zip(R.auc,R.rate)): ax.text(a+0.01,i,f'AUC {a:.2f}  (결측률 {r:.0%})',va='center',fontsize=11,color=INK)
ax.set_xlim(0.4,1.15); ax.set_xticks([0.4,0.5,0.6,0.7,0.8,0.9,1.0]); ax.set_ylim(-0.5,len(R)-0.1); ax.set_xlabel('결측 여부 예측 AUC',color=GREY,fontsize=11)
fig.tight_layout(); fig.savefig('eda_missing.png',facecolor='white'); plt.close()
print('threshold first/last', round(T.iloc[0],2), round(T.iloc[-1],2)); print(R.round(3))
